In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
with np.load(INPUT / 'tokens.npz') as data:
    tokens = {key: data[key] for key in data.files}
device = device_for(os.environ.get('NEURAL_DNA_DEVICE', 'auto'))
model = load_model(INPUT / 'model', device)
base = load_file(str(INPUT / 'model/model.safetensors'))
layout = marker_layout(model, max(cfg['marker_counts']), rng)
assert layout['maximum'] <= layout['parameters_per_block']
(OUTPUT / 'marker_layout.json').write_text(json.dumps(layout))
donor_directory = OUTPUT / 'donors'
donor_directory.mkdir(exist_ok=True)

In [ ]:
panels = {'base': fingerprint(model, layout)}
rows = [{'role': 'pretrained' if cfg['backend'] == 'pretrained' else 'random_fixture', 'source': -1,
         **language_metrics(model, tokens['evaluation_ids'])}]
exact_cal, exact_test, proxy = [], [], []
for source in range(cfg['sources']):
    reset_blocks(model, base)
    train_steps(model, tokens[f'family{source}'], cfg['family_steps'], rng, cfg)
    family = block_state(model)
    siblings = []
    roles = [f'reference{j}' for j in range(cfg['references_per_source'])] + ['cal', 'test']
    for role in roles:
        reset_blocks(model, family)
        train_steps(model, tokens[f'family{source}'], cfg['child_steps'], rng, cfg)
        if role in ['cal', 'test']:
            checkpoint = donor_directory / f'{role}_{source}.safetensors'
            save_delta(model, base, checkpoint)
            apply_delta(model, base, checkpoint)
        values = fingerprint(model, layout)
        if role.startswith('reference'):
            siblings.append(values)
        elif role == 'cal':
            exact_cal.append(values[None])
        else:
            exact_test.append(values[None])
        rows.append({'role': role, 'source': source, **language_metrics(model, tokens['evaluation_ids'])})
    proxy.append(siblings)
    del family
panels.update(exact_cal=np.stack(exact_cal), exact_test=np.stack(exact_test), proxy=np.array(proxy))
np.savez_compressed(OUTPUT / 'source_fingerprints.npz', **panels)
pd.DataFrame(rows).to_csv(OUTPUT / 'source_metrics.csv', index=False)
training_info = {'device': str(device), 'model_parameters': sum(p.numel() for p in model.parameters()),
    'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
    'markers_per_block': layout['maximum'], 'full_parameters_per_block': layout['parameters_per_block'],
    'checkpoint_delta_dtype': 'float16', 'frozen': 'embeddings, tied output head, final norm'}
(OUTPUT / 'training_info.json').write_text(json.dumps(training_info, indent=2))
del model, base
release_memory()
display(pd.DataFrame(rows))